# Data Pipeline for Senate LDA Expenses
---

This is a data pipe designed to be used to pull out and sum all Senate LDA expense data while removing duplicates created by amendments or double-filling. The data at the end is very similar to Open secrets data, though the Open Secrets lobbying page shows totals that include the income statements from lobbying firms and I don't know much about those so I've left them out to be added at some future date. 

## Input Requirements
- To run this pipeline you will need to first get a csv from lobbyfinder.py
- This pipeline is designed to use all of the inbuild column names from lobbyfinder.py so I don't think it will work with anything else

## To Use
- Make sure all requirements are installed, as I installed the ones I needed as I went
- import your .csv into this folder and change the pdf.readcsv to your .csv name


## Output
- This will leave you with 4 graphics in the Viz folder all built with ploty so that you can interact with them in the notebook to check values and dates. 

### Libraries Used
- pandas, plotly, kaledio, nbformat
---
### Last Update: 04/26/26
---

## Step 1: dependances and importing the data

In [19]:
%pip install plotly

import sys
!{sys.executable} -m pip install kaleido

import plotly.io as pio
pio.kaleido.scope.mathjax = None



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\craig\AppData\Local\Temp\ipykernel_10324\1501935392.py:7: DeprecationWarning: 
Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.

  pio.kaleido.scope.mathjax = None


In [20]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [21]:
df = pd.read_csv('lobbying_filings_Mega.csv')
df.head()

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url
0,Senate LDA,OpenAI,client,5d928e9c-a1a2-4688-baf6-837d1abb6492,2023,NaN,4th Quarter - Report,2024-01-22T16:53:42-05:00,"OPENAI OPCO, LLC",401107987,"OPENAI OPCO, LLC",57805,NaN,260000.0,Computer Industry; Copyright/Patent/Trademark;...,https://lda.senate.gov/filings/public/filing/5...
1,Senate LDA,OpenAI,client,cbad7c81-7058-45fc-8658-2436ce676257,2023,NaN,4th Quarter - Report,2024-01-22T18:11:55-05:00,DLA PIPER LLP (US),76855,"OPENAI OPCO, LLC",57807,90000.0,NaN,Computer Industry,https://lda.senate.gov/filings/public/filing/c...
2,Senate LDA,OpenAI,client,7776e176-d3d3-48c1-8921-37c0c5d35cce,2023,NaN,4th Quarter - Report (No Activity),2024-01-19T10:31:46-05:00,AKIN GUMP STRAUSS HAUER & FELD,682,"OPENAI OPCO, LLC",58080,30000.0,NaN,NaN,https://lda.senate.gov/filings/public/filing/7...
3,Senate LDA,OpenAI,client,aa0fcd21-baea-4258-a87c-2fb384b7c454,2024,NaN,4th Quarter - Report (No Activity),2025-01-13T16:00:22-05:00,HOGAN LOVELLS US LLP,18422,"OPENAI OPCO, LLC",59054,NaN,NaN,NaN,https://lda.senate.gov/filings/public/filing/a...
4,Senate LDA,OpenAI,client,852b11b5-898d-45a7-95c8-d30950b9440a,2024,NaN,4th Quarter - Report,2025-01-21T10:50:36-05:00,AKIN GUMP STRAUSS HAUER & FELD,682,"OPENAI OPCO, LLC",58080,80000.0,NaN,Computer Industry,https://lda.senate.gov/filings/public/filing/8...


## Step 2 - Data Cleaning

In [22]:
# Filter for [role] = registrant 

df_clean = df[df['role'] == 'registrant']


In [23]:
# Gather Duplicate Quarter reports & Amendments

dupes = df_clean[df_clean.duplicated(subset=['client_name', 'filing_year', 'filing_type'], keep=False)]
dupes[['client_name', 'filing_year', 'filing_type', 'dt_posted']].sort_values(['client_name', 'filing_year', 'filing_type'])


,client_name,filing_year,filing_type,dt_posted
442,AEROVIRONMENT INC.,2009,2nd Quarter - Report,2009-07-20T18:15:50-04:00
443,AEROVIRONMENT INC.,2009,2nd Quarter - Report,2009-07-20T18:39:39-04:00
451,AEROVIRONMENT INC.,2011,2nd Quarter - Report,2011-07-18T10:55:45.337000-04:00
452,AEROVIRONMENT INC.,2011,2nd Quarter - Report,2011-10-18T16:11:33.507000-04:00


In [24]:
# View Total year for identified dups

df_clean[(df_clean['client_name'] == 'AEROVIRONMENT INC.') & (df_clean['filing_year'] == 2009)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url
440,Senate LDA,aerovironment,registrant,63b48082-2b18-4f38-81bb-e4f06ec2a8b2,2009,NaN,Registration,2009-06-16T15:25:09-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,NaN,Transportation; Energy/Nuclear; Homeland Secur...,https://lda.senate.gov/filings/public/filing/6...
441,Senate LDA,aerovironment,registrant,ca80207c-22f7-4aec-81fb-7e8bcce84268,2009,NaN,1st Quarter - Report,2009-06-16T17:07:08-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,320000.0,Defense,https://lda.senate.gov/filings/public/filing/c...
442,Senate LDA,aerovironment,registrant,0c8799f6-e2dd-40fe-ac76-965ed9dc3d7b,2009,NaN,2nd Quarter - Report,2009-07-20T18:15:50-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,120000.0,Defense,https://lda.senate.gov/filings/public/filing/0...
443,Senate LDA,aerovironment,registrant,40f0d4eb-474d-47ab-b589-0d985e4a320f,2009,NaN,2nd Quarter - Report,2009-07-20T18:39:39-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,120000.0,Defense,https://lda.senate.gov/filings/public/filing/4...
444,Senate LDA,aerovironment,registrant,317ed170-04fd-47af-b1aa-f9154751f44d,2009,NaN,3rd Quarter - Report,2009-10-20T16:47:31-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,180000.0,Defense,https://lda.senate.gov/filings/public/filing/3...
445,Senate LDA,aerovironment,registrant,b06a7978-f42a-45b7-88b0-6929a169f987,2009,NaN,4th Quarter - Report,2010-01-21T14:40:05.597000-05:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,133000.0,Defense,https://lda.senate.gov/filings/public/filing/b...


In [25]:
# Creating a new column for base_quarter, and removing dups based on dt_posted datetime

df_clean['base_quarter'] = df_clean['filing_type'].str.replace(r'\s*-\s*(Report|Amendment).*', '', regex=True).str.strip()

df_deduped = (
    df_clean.sort_values('dt_posted', ascending=False)
            .drop_duplicates(subset=['client_name', 'filing_year', 'base_quarter'], keep='first')
            .reset_index(drop=True)
)


In [26]:
df_deduped[(df_deduped['client_name'] == 'AEROVIRONMENT INC.') & (df_deduped['filing_year'] == 2010)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url,base_quarter
173,Senate LDA,aerovironment,registrant,466bd7f2-33cc-4f43-b833-d709d046dfb9,2010,NaN,4th Quarter - Report,2011-01-18T20:07:15.537000-05:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,145000.0,Defense,https://lda.senate.gov/filings/public/filing/4...,4th Quarter
175,Senate LDA,aerovironment,registrant,e8b9b724-9e8e-4e1a-afa7-717231ef39d0,2010,NaN,3rd Quarter - Report,2010-10-08T14:46:16.263000-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,145000.0,Defense,https://lda.senate.gov/filings/public/filing/e...,3rd Quarter
177,Senate LDA,aerovironment,registrant,e735b155-03ec-408d-95be-d1760043a767,2010,NaN,2nd Quarter - Report,2010-07-19T13:00:23.140000-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,160000.0,Defense,https://lda.senate.gov/filings/public/filing/e...,2nd Quarter
178,Senate LDA,aerovironment,registrant,10471c7f-0e32-4fb8-98a7-e8d155612b0f,2010,NaN,1st Quarter - Report,2010-04-20T12:36:12.673000-04:00,"AEROVIRONMENT, INC.",400454169,AEROVIRONMENT INC.,191038,NaN,127000.0,Defense,https://lda.senate.gov/filings/public/filing/1...,1st Quarter


In [27]:
# Sum the expenses for the deduped years

df_deduped.groupby('filing_year')['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})



,filing_year,expenses
0,2009,"$753,000"
1,2010,"$837,000"
2,2011,"$1,035,000"
3,2012,"$490,000"
4,2013,"$1,242,000"
5,2014,"$880,000"
6,2015,"$1,142,000"
7,2016,"$1,160,000"
8,2017,"$1,710,000"
9,2018,"$1,830,000"


In [28]:
df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})


,client_name,filing_year,expenses
0,AEROVIRONMENT INC.,2009,"$753,000"
1,AEROVIRONMENT INC.,2010,"$577,000"
2,AEROVIRONMENT INC.,2011,"$485,000"
3,AEROVIRONMENT INC.,2012,"$290,000"
4,AEROVIRONMENT INC.,2013,"$122,000"
5,AEROVIRONMENT INC.,2014,"$150,000"
6,AEROVIRONMENT INC.,2015,"$142,000"
7,AEROVIRONMENT INC.,2016,"$140,000"
8,AEROVIRONMENT INC.,2017,"$300,000"
9,AEROVIRONMENT INC.,2018,"$170,000"


## Step 3 - Vizualizations 

In [29]:
# installing additional dependancies 

import sys
!{sys.executable} -m pip install --upgrade nbformat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# Graphing each companies summed expenses per year

df_yearly = df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index()

fig = px.line(df_yearly, x='filing_year', y='expenses', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'expenses': 'Lobbying Expenses', 'client_name': 'Company'})
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='Total Company Lobbying Expenses by Year')

fig.show()

fig.write_image('Viz/Company_change_expenses_by_year.svg', scale=2)



In [31]:
# creating the % change dataset

df_pct = (
    df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum()
    .reset_index()
    .sort_values(['client_name', 'filing_year'])
)

df_pct['pct_change'] = df_pct.groupby('client_name')['expenses'].pct_change() * 100
df_pct['pct_change'] = df_pct['pct_change'].fillna(0)


df_pct.head(20)


,client_name,filing_year,expenses,pct_change
0,AEROVIRONMENT INC.,2009,753000.0,0.000000
1,AEROVIRONMENT INC.,2010,577000.0,-23.373174
2,AEROVIRONMENT INC.,2011,485000.0,-15.944541
3,AEROVIRONMENT INC.,2012,290000.0,-40.206186
4,AEROVIRONMENT INC.,2013,122000.0,-57.931034
5,AEROVIRONMENT INC.,2014,150000.0,22.950820
6,AEROVIRONMENT INC.,2015,142000.0,-5.333333
7,AEROVIRONMENT INC.,2016,140000.0,-1.408451
8,AEROVIRONMENT INC.,2017,300000.0,114.285714
9,AEROVIRONMENT INC.,2018,170000.0,-43.333333


In [32]:
# Graphing %change by company

fig = px.line(df_pct, x='filing_year', y='pct_change', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'pct_change': '% Change in Expenses', 'client_name': 'Company'})
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='% Change in Lobbying Expenses by Year')

fig.write_image('Viz/%_change_expenses_by_year.svg', scale=2)

fig.show()


In [33]:
# showing totals

df_total = df_deduped.groupby('filing_year')['expenses'].sum().reset_index()
df_total['pct_change'] = df_total['expenses'].pct_change() * 100
df_total['pct_change'] = df_total['pct_change'].fillna(0)

fig1 = px.line(df_total, x='filing_year', y='expenses', markers=True,
               title='Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'expenses': 'Total Expenses'})
fig1.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig1.show()

fig2 = px.line(df_total, x='filing_year', y='pct_change', markers=True,
               title='% Change in Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'pct_change': '% Change'})
fig2.add_hline(y=0, line_dash='dash', line_color='grey')
fig2.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))


fig1.write_image('Viz/total_expenses_by_year.svg', scale=2)
fig2.write_image('Viz/pct_change_total_expenses.svg', scale=2)

fig2.show()